In [ ]:
import pandas as pd
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dhruvildave/en-fr-translation-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'en-fr-translation-dataset' dataset.
Path to dataset files: /kaggle/input/en-fr-translation-dataset


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'language-translation-englishfrench' dataset.
Path to dataset files: /kaggle/input/language-translation-englishfrench


In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        l = os.path.join(root, file)


In [ ]:

import pandas as pd
file_path = os.path.join(path, l)
df = pd.read_csv(file_path)
df.head()
df = df.sample(
    n=20000,
    random_state=42
).reset_index(drop=True)

In [ ]:
print(df.isnull().sum())

English words/sentences    0
French words/sentences     0
dtype: int64


In [ ]:
import re
def tokenize(sent):
  sent = sent.lower()
  sent = re.sub(r"([.,!?;:])", r" \1", sent)
  return sent.split()

In [ ]:
t  = df["English words/sentences"][0]

In [ ]:
tokenize(t)

['take', 'a', 'seat', '.']

In [ ]:
from collections import Counter
#Counter(df["English words/sentences"])

class Vocab:
  def __init__(self):
    self.word2idx = {
        "<PAD>" : 0,
        "<UNK>" : 1,
        "<SOS>" : 2,
        "<EOS>" : 3
    }

    self.idx2word = {
        0 : "<PAD>",
        1 : "<UNK>",
        2 : "<SOS>",
        3: "<EOS>",

    }
  def build(self, sents, min_freq = 1):
    counter = Counter()
    for sent in sents.tolist():
      t = tokenize(sent)
      counter.update(t)

    for word, freq in counter.items():
      if freq >= min_freq:
        idx = len(self.word2idx)
        self.word2idx[word] = idx
        self.idx2word[idx] = word

    def numericalize(self, sent):
      tokens = tokenize(sent)
      return [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens]

    def __len__(self):
      return len(self.word2idx)

In [ ]:
df

,English words/sentences,French words/sentences
0,Take a seat.,Prends place !
1,I wish Tom was here.,J'aimerais que Tom soit là.
2,How did the audition go?,Comment s'est passée l'audition ?
3,I've no friend to talk to about my problems.,Je n'ai pas d'ami avec lequel je puisse m'entr...
4,I really like this skirt. Can I try it on?,"J'aime beaucoup cette jupe, puis-je l'essayer ?"
...,...,...
19995,Plants grow.,Les plantes croissent.
19996,You look like an imbecile.,Tu as l'air d'un imbécile.
19997,Are you ready now?,Êtes-vous prêts maintenant ?
19998,They accused me of being a liar.,Elles m'ont accusé d'être un menteur.


In [ ]:
eng_vocab = Vocab()
fre_vocab = Vocab()

eng_vocab.build(df["English words/sentences"])
fre_vocab.build(df["French words/sentences"])

In [ ]:
print("\nEnglish:")
print(list(eng_vocab.word2idx.items())[:20])

print("\nFrench:")
print(list(fre_vocab.word2idx.items())[:20])


English:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('take', 4), ('a', 5), ('seat', 6), ('.', 7), ('i', 8), ('wish', 9), ('tom', 10), ('was', 11), ('here', 12), ('how', 13), ('did', 14), ('the', 15), ('audition', 16), ('go', 17), ('?', 18), ("i've", 19)]

French:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('prends', 4), ('place', 5), ('!', 6), ("j'aimerais", 7), ('que', 8), ('tom', 9), ('soit', 10), ('là', 11), ('.', 12), ('comment', 13), ("s'est", 14), ('passée', 15), ("l'audition", 16), ('?', 17), ('je', 18), ("n'ai", 19)]


In [ ]:
from collections import Counter
#Counter(df["English words/sentences"])

class Vocab:
  def __init__(self):
    self.word2idx = {
        "<PAD>" : 0,
        "<UNK>" : 1,
        "<SOS>" : 2,
        "<EOS>" : 3
    }

    self.idx2word = {
        0 : "<PAD>",
        1 : "<UNK>",
        2 : "<SOS>",
        3: "<EOS>",

    }
  def build(self, sents, min_freq = 1):
    counter = Counter()
    for sent in sents.tolist():
      t = tokenize(sent)
      counter.update(t)

    for word, freq in counter.items():
      if freq >= min_freq:
        idx = len(self.word2idx)
        self.word2idx[word] = idx
        self.idx2word[idx] = word

  def numericalize(self, sent):
      tokens = tokenize(sent)
      return ( [self.word2idx["<SOS>"]] + [self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens] + [self.word2idx["<EOS>"]])
  def __len__(self):
      return len(self.word2idx)

In [ ]:
english_vocab = Vocab()
french_vocab = Vocab()

english_vocab.build(df["English words/sentences"])
french_vocab.build(df["French words/sentences"])

print("\nEnglish:")
print(list(english_vocab.word2idx.items())[:20])

print("\nFrench:")
print(list(french_vocab.word2idx.items())[:20])


English:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('take', 4), ('a', 5), ('seat', 6), ('.', 7), ('i', 8), ('wish', 9), ('tom', 10), ('was', 11), ('here', 12), ('how', 13), ('did', 14), ('the', 15), ('audition', 16), ('go', 17), ('?', 18), ("i've", 19)]

French:
[('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('prends', 4), ('place', 5), ('!', 6), ("j'aimerais", 7), ('que', 8), ('tom', 9), ('soit', 10), ('là', 11), ('.', 12), ('comment', 13), ("s'est", 14), ('passée', 15), ("l'audition", 16), ('?', 17), ('je', 18), ("n'ai", 19)]


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size= 0.2, random_state = 42)



In [ ]:
df["English words/sentences"]
df["French words/sentences"]

,French words/sentences
0,Prends place !
1,J'aimerais que Tom soit là.
2,Comment s'est passée l'audition ?
3,Je n'ai pas d'ami avec lequel je puisse m'entr...
4,"J'aime beaucoup cette jupe, puis-je l'essayer ?"
...,...
19995,Les plantes croissent.
19996,Tu as l'air d'un imbécile.
19997,Êtes-vous prêts maintenant ?
19998,Elles m'ont accusé d'être un menteur.


In [ ]:
import torch
from torch.utils.data import Dataset


class TranslateDataset(Dataset):
  def __init__(self, df, english_vocab, french_vocab):
    self.data = df.reset_index(drop = True)

    self.eng = english_vocab
    self.fre = french_vocab

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    en = self.data.iloc[index]["English words/sentences"]
    fr = self.data.iloc[index]["French words/sentences"]

    engl_ids = self.eng.numericalize(en)
    fren_ids = self.fre.numericalize(fr)

    return (
        torch.tensor(engl_ids, dtype = torch.long),
        torch.tensor(fren_ids, dtype = torch.long)
    )


In [ ]:
train_data = TranslateDataset(
    train_df, english_vocab, french_vocab
)

In [ ]:
train_dataset = TranslateDataset(
    train_df,
    english_vocab,
    french_vocab
)

val_dataset = TranslateDataset(
    val_df,
    english_vocab,
    french_vocab
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 16000
Validation dataset: 4000


In [ ]:
src, trg = train_dataset[0]

print("Source:", src)
print("Target:", trg)

Source: tensor([  2, 646,  35,  41, 284, 577,   7,   3])
Target: tensor([   2,   52,  198,    8,  119, 3715,  476,   12,    3])


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def padding(batch):
  input = []
  label = []

  for i, l in batch:
    input.append(i)
    label.append(l)

  input = pad_sequence( input, batch_first= True, padding_value= english_vocab.word2idx["<PAD>"])
  label = pad_sequence( label, batch_first= True, padding_value= french_vocab.word2idx["<PAD>"])

  return input, label

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn= padding
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn= padding
)

In [ ]:
src, trg = next(iter(train_loader))

print("Source shape:", src.shape)
print("Target shape:", trg.shape)

print("\nSource batch:")
print(src)

print("\nTarget batch:")
print(trg)

Source shape: torch.Size([32, 19])
Target shape: torch.Size([32, 20])

Source batch:
tensor([[   2,  208,   41,  284, 2403,   18,    3,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   35,  253,   41,  216,   12,   75,   18,    3,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   10,  255,    5,   73,  748,    7,    3,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   85,   41,  253,    5,  265,   18,    3,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   86,  332,   22,   77,    5,   56,    7,    3,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   47,   15,  634, 4400,  399,   18,    3,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [   2,   90,  560,   22,  585,   68,  168,   90, 1646,   75,    5, 1198,
            7,    3,    0,    0,    

In [ ]:
import re
from collections import Counter

In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
  def __init__(self,input_dim,  embedding_dim, hidden_dim,num_layer = 1, dropout = 0.0 ):
    super().__init__()

    self.embedding = nn.Embedding(input_dim, embedding_dim, padding_idx= english_vocab.word2idx["<PAD>"])
    self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers = num_layer, dropout= dropout if num_layer > 1 else 0)
    self.dropout = nn.Dropout(dropout)

  def forward(self, src):
    x = self.dropout(self.embedding(src))
    x = x.permute(1, 0, 2)

    op, hid = self.rnn(x)

    return op, hid

In [ ]:
INPUT_DIM = len(english_vocab)
OUTPUT_DIM = len(french_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
ATTENTION_DIM = 64

encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=ENC_EMB_DIM,
    hidden_dim=HID_DIM
)

print(encoder)

src, trg = next(iter(train_loader))

print("Source shape:", src.shape)

op, hidden = encoder(src)

print("Hidden shape:", hidden.shape)



Encoder(
  (embedding): Embedding(6541, 128, padding_idx=0)
  (rnn): GRU(128, 256)
  (dropout): Dropout(p=0.0, inplace=False)
)
Source shape: torch.Size([32, 19])
Hidden shape: torch.Size([1, 32, 256])


In [ ]:
class Attention(nn.Module):

    def __init__(self, encoder_hidden_dim, decoder_hidden_dim, attention_dim):
        super().__init__()

        self.attn = nn.Linear( encoder_hidden_dim + decoder_hidden_dim, attention_dim)
        self.v = nn.Linear( attention_dim, 1, bias=False )

    def forward(self, hidden, encoder_outputs):
        hidden = hidden[-1]
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1)
        hidden = hidden.repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh( self.attn( torch.cat( (hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

In [ ]:
class Decoder(nn.Module):
  def __init__(self, output_dim, embedding_dim, encoder_hidden_dim, decoder_hidden_dim, attention, num_layers = 1, dropout = 0.0):
    super().__init__()
    self.output_dim = output_dim
    self.embedding = nn.Embedding(output_dim, embedding_dim, padding_idx= french_vocab.word2idx["<PAD>"])
    self.attention = attention
    self.rnn = nn.GRU(
        embedding_dim + encoder_hidden_dim, decoder_hidden_dim, num_layers, dropout= dropout if num_layers > 1 else 0
    )

    self.fc = nn.Linear(embedding_dim
            + encoder_hidden_dim
            + decoder_hidden_dim, output_dim)

    self.dropout = nn.Dropout(dropout)


  def forward(self, input, hidden, encoder_outputs):
    input = input.unsqueeze(0)
    embedded = self.dropout(self.embedding(input))

    attention = self.attention(
        hidden,
        encoder_outputs
    )
    attention = attention.unsqueeze(1)
    encoder_outputs = encoder_outputs.permute(1, 0, 2)
    context = torch.bmm(
        attention,
        encoder_outputs
    )

    context = context.permute(
        1, 0, 2
    )
    """
    print("input:", input.shape)
    print("embedded:", embedded.shape)
    print("attention:", attention.shape)
    print("encoder_outputs:", encoder_outputs.shape)
    print("context:", context.shape)"""

    rnn_input = torch.cat(
        (embedded, context),
        dim=2
    )

    output, hidden = self.rnn(
        rnn_input,
        hidden
    )
    prediction = self.fc(
        torch.cat(
            (output, context, embedded),
            dim=2
        )
    )

    return prediction.squeeze(0), hidden

In [ ]:
src, trg = next(iter(train_loader))

encoder_outputs, hidden = encoder(src)

print("Hidden:", hidden)

Hidden: tensor([[[-0.0459,  0.0093, -0.0140,  ..., -0.0355, -0.0351, -0.0127],
         [-0.0468,  0.0093, -0.0144,  ..., -0.0347, -0.0345, -0.0147],
         [-0.0460,  0.0090, -0.0140,  ..., -0.0355, -0.0350, -0.0128],
         ...,
         [-0.0457,  0.0088, -0.0140,  ..., -0.0361, -0.0349, -0.0118],
         [-0.0458,  0.0088, -0.0141,  ..., -0.0363, -0.0349, -0.0117],
         [-0.0458,  0.0087, -0.0141,  ..., -0.0363, -0.0349, -0.0117]]],
       grad_fn=<StackBackward0>)


In [ ]:
input_token = trg[:, 0]
input_token.shape

torch.Size([32])

In [ ]:
"""prediction, hidden = decoder( input_token, hidden,
    encoder_outputs)"""

'prediction, hidden = decoder( input_token, hidden,\n    encoder_outputs)'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

INPUT_DIM = len(english_vocab)
OUTPUT_DIM = len(french_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
ATTENTION_DIM = 64


# -------------------------
# 1. Encoder
# -------------------------

encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=ENC_EMB_DIM,
    hidden_dim=HID_DIM
)



attention = Attention(
    encoder_hidden_dim=HID_DIM,
    decoder_hidden_dim=HID_DIM,
    attention_dim=ATTENTION_DIM
)


decoder = Decoder(
    output_dim=OUTPUT_DIM,
    embedding_dim=DEC_EMB_DIM,
    encoder_hidden_dim=HID_DIM,
    decoder_hidden_dim=HID_DIM,
    attention=attention
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [ ]:

class SeqtoSeq(nn.Module):
  def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device
  def forward(self, src, trg, teacher_forcing_ratio=0.5) :
    batch_size = src.shape[0]
    trg_len = trg.shape[1]
    trg_vocab_size = self.decoder.output_dim
    outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
    encoder_outputs, hidden = self.encoder(src)
    input = trg[:, 0] #1st decoder input
    for t in range(1, trg_len):
      output, hidden = self.decoder(input, hidden, encoder_outputs)
      outputs[:, t, :] = output
      top1 = output.argmax(1)
      teacher_force = (
                torch.rand(1).item() < teacher_forcing_ratio
            )
      if teacher_force:
                input = trg[:, t]
      else:
                input = top1

    return outputs


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
model = SeqtoSeq( encoder, decoder, device).to(device)
print(model)

import torch.optim as optim

PAD_IDX = french_vocab.word2idx["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam( model.parameters(), lr=0.001)

Device: cpu
SeqtoSeq(
  (encoder): Encoder(
    (embedding): Embedding(6541, 128, padding_idx=0)
    (rnn): GRU(128, 256)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(11050, 128, padding_idx=0)
    (attention): Attention(
      (attn): Linear(in_features=512, out_features=64, bias=True)
      (v): Linear(in_features=64, out_features=1, bias=False)
    )
    (rnn): GRU(384, 256)
    (fc): Linear(in_features=640, out_features=11050, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


In [ ]:
model = SeqtoSeq(encoder, decoder,device).to(device)
print(model)


PAD_IDX = french_vocab.word2idx["<PAD>"]
criterion = nn.CrossEntropyLoss( ignore_index=PAD_IDX )

optimizer = optim.Adam( model.parameters(), lr=0.001)

SeqtoSeq(
  (encoder): Encoder(
    (embedding): Embedding(6541, 128, padding_idx=0)
    (rnn): GRU(128, 256)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(11050, 128, padding_idx=0)
    (attention): Attention(
      (attn): Linear(in_features=512, out_features=64, bias=True)
      (v): Linear(in_features=64, out_features=1, bias=False)
    )
    (rnn): GRU(384, 256)
    (fc): Linear(in_features=640, out_features=11050, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


In [ ]:
from tqdm import tqdm

def train(model, iterator, optimizer, criterion, clip=1):

    model.train()
    epoch_loss = 0

    progress = tqdm(iterator, desc="Training")

    for src, trg in progress:

        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()
        output = model(src,trg,teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]
        output = output[:, 1:, :].contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_( model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()
        progress.set_postfix(
            loss=f"{loss.item():.3f}"
        )

    return epoch_loss / len(iterator)

In [ ]:
'''
def train(model, iterator, optimizer, criterion, clip=1):

    model.train()

    epoch_loss = 0

    for src, trg in iterator:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]
        output = output[:, 1:, :].contiguous().view(
            -1,
            output_dim
        )

        trg = trg[:, 1:].contiguous().view(-1)

        loss = criterion(
            output,
            trg
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            clip
        )

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)
'''
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for src, trg in iterator:

            src = src.to(device)
            trg = trg.to(device)

            output = model(
                src,
                trg,
                teacher_forcing_ratio=0
            )

            output_dim = output.shape[-1]

            output = output[:, 1:, :].contiguous().view(
                -1,
                output_dim
            )

            trg = trg[:, 1:].contiguous().view(-1)

            loss = criterion(
                output,
                trg
            )

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for src, trg in iterator:

            src = src.to(device)
            trg = trg.to(device)

            output = model(src,trg,teacher_forcing_ratio=0.0)
            output_dim = output.shape[-1]

            output = output[:, 1:, :].contiguous().view(-1,output_dim)
            trg = trg[:, 1:].contiguous().view(-1)

            loss = criterion(output,trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
import math
import time
loss_train_att = []
loss_val_att = []
N_EPOCHS = 5

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(
        model,
        train_loader,
        optimizer,
        criterion
    )

    valid_loss = evaluate(
        model,
        val_loader,
        criterion
    )
    loss_train_att.append(train_loss)
    loss_val_att.append(valid_loss)
    end_time = time.time()

    epoch_mins = int((end_time - start_time) // 60)
    epoch_secs = int((end_time - start_time) % 60)

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            model.state_dict(),
            "translation_model_attention.pt"
        )

    print(
        f"Epoch: {epoch+1:02} | "
        f"Time: {epoch_mins}m {epoch_secs}s"
    )

    print(
        f"\tTrain Loss: {train_loss:.3f}"
    )

    print(
        f"\tVal. Loss: {valid_loss:.3f}"
    )

model.load_state_dict(
    torch.load(
        "translation_model_attention.pt",
        map_location=device
    )
)

Training: 100%|██████████| 500/500 [10:30<00:00,  1.26s/it, loss=4.050]


Epoch: 01 | Time: 11m 14s
	Train Loss: 4.799
	Val. Loss: 4.349


Training: 100%|██████████| 500/500 [10:04<00:00,  1.21s/it, loss=3.160]


Epoch: 02 | Time: 10m 50s
	Train Loss: 3.203
	Val. Loss: 3.865


Training: 100%|██████████| 500/500 [10:03<00:00,  1.21s/it, loss=2.617]


...

Training: 100%|██████████| 500/500 [09:50<00:00,  1.18s/it, loss=1.739]


Epoch: 04 | Time: 10m 33s
	Train Loss: 1.742
	Val. Loss: 3.730


Training: 100%|██████████| 500/500 [09:57<00:00,  1.19s/it, loss=1.386]


Epoch: 05 | Time: 10m 41s
	Train Loss: 1.347
	Val. Loss: 3.821


<All keys matched successfully>

In [ ]:
def translate_sentence(
    sentence,
    model,
    english_vocab,
    french_vocab,
    device,
    max_len=50
):

    model.eval()

    tokens = tokenize(sentence)

    tokens = [
        "<SOS>"
    ] + tokens + [
        "<EOS>"
    ]

    src_indexes = [
        english_vocab.word2idx.get(
            token,
            english_vocab.word2idx["<UNK>"]
        )
        for token in tokens
    ]

    src_tensor = torch.tensor(
        src_indexes,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    input_token = torch.tensor(
        [french_vocab.word2idx["<SOS>"]],
        dtype=torch.long
    ).to(device)

    translated_tokens = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden = model.decoder(
                input_token,
                hidden,
                encoder_outputs
            )

        predicted_token = output.argmax(1).item()

        if predicted_token == french_vocab.word2idx["<EOS>"]:
            break

        translated_tokens.append(
            french_vocab.idx2word[predicted_token]
        )

        input_token = torch.tensor(
            [predicted_token],
            dtype=torch.long
        ).to(device)

    return translated_tokens

print(loss_train_att)
print(loss_val_att )

[4.799361021518707, 3.2030399584770204, 2.3319017128944397, 1.7419939544200898, 1.3465655380487442]
[4.3485455856323245, 3.8652354259490966, 3.7660784397125244, 3.7298819770812988, 3.8205104007720947]


In [ ]:
print(loss_train_att)
print(loss_val_att )

[4.799361021518707, 3.2030399584770204, 2.3319017128944397, 1.7419939544200898, 1.3465655380487442]
[4.3485455856323245, 3.8652354259490966, 3.7660784397125244, 3.7298819770812988, 3.8205104007720947]
